# Imagebits



In [ ]:
# =========================
# 1) Imports & Config
# =========================
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, f1_score

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

%matplotlib inline


In [ ]:
@dataclass
class CFG:
    # --- data ---
    DATA_ROOT: Path = Path(".")
    IMAGEBITS_DIR: Path = Path("imagebits")
    VAL_RATIO: float = 0.2

    # --- runtime ---
    SEED: int = 42
    IMG_SIZE: int = 96
    BATCH_SIZE: int = 128
    NUM_WORKERS: int = 2
    PIN_MEMORY: bool = True

    # --- training ---
    EPOCHS: int = 15
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4

cfg = CFG()

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # reproducibilitate
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 2) Paths + Meta datasets
# =========================
imagebits_train_dir = cfg.DATA_ROOT / cfg.IMAGEBITS_DIR / "train"
imagebits_test_dir  = cfg.DATA_ROOT / cfg.IMAGEBITS_DIR / "test"

assert imagebits_train_dir.exists(), f"Missing: {imagebits_train_dir}"
assert imagebits_test_dir.exists(),  f"Missing: {imagebits_test_dir}"

# Meta (fără transformări) doar pentru targets/clase/număr
train_meta = datasets.ImageFolder(root=str(imagebits_train_dir), transform=None)
test_meta  = datasets.ImageFolder(root=str(imagebits_test_dir),  transform=None)

raw_class_folders = train_meta.classes
num_classes = len(raw_class_folders)

CIFAR10_NAMES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]

def try_human_names(folders: List[str]):
    if len(folders) == 10 and all(f.isdigit() for f in folders):
        nums = [int(f) for f in folders]
        if sorted(nums) == list(range(10)):
            mapping = {str(i): CIFAR10_NAMES[i] for i in range(10)}
            return [mapping[f] for f in folders]
        if sorted(nums) == list(range(1, 11)):
            mapping = {str(i+1): CIFAR10_NAMES[i] for i in range(10)}
            return [mapping[f] for f in folders]
    return folders

class_names = try_human_names(raw_class_folders)

print("Raw class folders:", raw_class_folders)
print("Class names used:", class_names)
print("Num classes:", num_classes)
print("Train size:", len(train_meta))
print("Test size:", len(test_meta))


## Data checks & EDA

- verificare fișiere corupte
- rezoluții
- mean/std pe canale (pentru normalizare)
- distribuția luminozității
- balans pe clase
- exemple vizuale per clasă


In [ ]:
# =========================
# 3) EDA utilities
# =========================

def list_image_files(root_dir: Path) -> List[Path]:
    root_dir = Path(root_dir)
    files = []
    for class_dir in root_dir.iterdir():
        if class_dir.is_dir():
            for p in class_dir.rglob("*"):
                if p.is_file():
                    files.append(p)
    return files


def scan_corrupted_images(root_dir: Path, max_report: int = 20):
    files = list_image_files(root_dir)
    bad = []
    ok = 0

    for p in tqdm(files, desc=f"Scanning corrupted: {root_dir}"):
        try:
            with Image.open(p) as img:
                img.verify()
            ok += 1
        except Exception as e:
            bad.append((p, str(e)))

    print(f"Total files scanned: {len(files)}")
    print(f"OK: {ok}")
    print(f"Corrupted/unreadable: {len(bad)}")

    if bad:
        print("Examples of bad files:")
        for p, err in bad[:max_report]:
            print(f"- {p} | {err}")

    return ok, bad


def resolution_stats(root_dir: Path, sample_limit: int = None):
    files = list_image_files(root_dir)
    if sample_limit is not None:
        files = files[:sample_limit]

    sizes = []
    for p in tqdm(files, desc=f"Reading resolutions: {root_dir}"):
        try:
            with Image.open(p) as img:
                sizes.append(img.size)  # (W, H)
        except Exception:
            pass

    cnt = Counter(sizes)
    print(f"Unique resolutions: {len(cnt)}")
    print("Top 10 resolutions:")
    for (w, h), c in cnt.most_common(10):
        print(f"  {w}x{h}: {c}")

    return sizes, cnt


def brightness_distribution(root_dir: Path, sample_limit: int = None):
    files = list_image_files(root_dir)
    if sample_limit is not None:
        files = files[:sample_limit]

    means = []
    for p in tqdm(files, desc=f"Brightness stats: {root_dir}"):
        try:
            with Image.open(p) as img:
                img = img.convert("L")
                arr = np.array(img, dtype=np.float32)
                means.append(arr.mean())
        except Exception:
            pass

    means = np.array(means)
    print(f"Images used: {len(means)}")
    print(f"Brightness mean: {means.mean():.2f}, std: {means.std():.2f}")
    return means


def compute_mean_std_imagefolder(root_dir: Path, img_size: int, batch_size: int, num_workers: int):
    """Mean/std pe canale din train: Resize + ToTensor (fără normalize, fără augmentări)."""
    tfm = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor()])
    ds = datasets.ImageFolder(root=str(root_dir), transform=tfm)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=False)

    n_pixels = 0
    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)

    for x, _ in tqdm(loader, desc=f"Computing mean/std: {root_dir}"):
        b, c, h, w = x.shape
        pixels = b * h * w
        n_pixels += pixels
        channel_sum += x.sum(dim=[0, 2, 3])
        channel_sum_sq += (x ** 2).sum(dim=[0, 2, 3])

    mean = channel_sum / n_pixels
    var = (channel_sum_sq / n_pixels) - mean ** 2
    std = torch.sqrt(var)

    return mean.numpy(), std.numpy()


def class_count_from_imagefolder(ds: datasets.ImageFolder) -> Dict[str, int]:
    counts = {c: 0 for c in ds.classes}
    for y in ds.targets:
        counts[ds.classes[y]] += 1
    return counts


def plot_bar_dict(d: Dict[str, int], title: str):
    plt.figure()
    plt.bar(list(d.keys()), list(d.values()))
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def show_examples_per_class(root_dir: Path, class_names: List[str], img_size: int, n_per_class: int = 3):
    vis_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])
    ds = datasets.ImageFolder(root=str(root_dir), transform=vis_tfms)

    idxs_by_class = {i: [] for i in range(len(class_names))}
    for idx, y in enumerate(ds.targets):
        if len(idxs_by_class[y]) < n_per_class:
            idxs_by_class[y].append(idx)
        if all(len(v) >= n_per_class for v in idxs_by_class.values()):
            break

    rows = len(class_names)
    cols = n_per_class
    plt.figure(figsize=(cols * 3, rows * 2.2))

    plot_i = 1
    for cls_idx, cls_name in enumerate(class_names):
        for j in range(n_per_class):
            ax = plt.subplot(rows, cols, plot_i)
            x, _ = ds[idxs_by_class[cls_idx][j]]
            ax.imshow(x.permute(1, 2, 0).clamp(0, 1))
            ax.set_axis_off()
            if j == 0:
                ax.set_title(cls_name)
            plot_i += 1

    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# 4) EDA run
# =========================

# 1) Corrupted scan
_ = scan_corrupted_images(imagebits_train_dir)
_ = scan_corrupted_images(imagebits_test_dir)

# 2) Resolutions
train_sizes, train_size_counts = resolution_stats(imagebits_train_dir)
_ , _ = resolution_stats(imagebits_test_dir)

top = train_size_counts.most_common(10)
labels = [f"{w}x{h}" for (w, h), _ in top]
values = [c for _, c in top]
plt.figure(figsize=(8, 3))
plt.bar(labels, values)
plt.title("Imagebits – Top train resolutions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3) Mean/std on train (for normalization)
mean_ib, std_ib = compute_mean_std_imagefolder(
    root_dir=imagebits_train_dir,
    img_size=cfg.IMG_SIZE,
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS
)
print("Train mean:", mean_ib)
print("Train std :", std_ib)

# 4) Brightness
b_train = brightness_distribution(imagebits_train_dir)
plt.figure(figsize=(6, 3))
plt.hist(b_train, bins=50)
plt.title("Imagebits – Train brightness distribution")
plt.xlabel("Mean intensity (0..255)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 5) Class balance (train/test)
train_counts = class_count_from_imagefolder(train_meta)
test_counts  = class_count_from_imagefolder(test_meta)
print("Train counts:", train_counts)
print("Test counts :", test_counts)
plot_bar_dict(train_counts, "Imagebits – Train class distribution")
plot_bar_dict(test_counts,  "Imagebits – Test class distribution")

# 6) Visual examples
show_examples_per_class(imagebits_train_dir, class_names, cfg.IMG_SIZE, n_per_class=2)


## Train/Val split (stratified) + DataLoaders

Facem split **stratificat** ca să păstrăm distribuția claselor în train/val.


In [ ]:
# =========================
# 5) Stratified split
# =========================

targets = np.array(train_meta.targets)
sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.VAL_RATIO, random_state=cfg.SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))
train_idx = train_idx.tolist()
val_idx   = val_idx.tolist()

print("Split sizes:", len(train_idx), len(val_idx))

def subset_class_counts_from_meta(meta_ds: datasets.ImageFolder, indices: List[int], raw_folders: List[str]) -> Dict[str, int]:
    counts = {c: 0 for c in raw_folders}
    for idx in indices:
        y = meta_ds.targets[idx]
        counts[raw_folders[y]] += 1
    return counts

train_split_counts = subset_class_counts_from_meta(train_meta, train_idx, raw_class_folders)
val_split_counts   = subset_class_counts_from_meta(train_meta, val_idx, raw_class_folders)

print("Train split counts:", train_split_counts)
print("Val split counts  :", val_split_counts)


In [ ]:
# =========================
# 6) Transforms + Datasets + Loaders
# =========================
MEAN = tuple(mean_ib.tolist())
STD  = tuple(std_ib.tolist())

train_tfms_base = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_tfms_aug = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

ds_train_base_full = datasets.ImageFolder(root=str(imagebits_train_dir), transform=train_tfms_base)
ds_train_aug_full  = datasets.ImageFolder(root=str(imagebits_train_dir), transform=train_tfms_aug)
ds_val_full        = datasets.ImageFolder(root=str(imagebits_train_dir), transform=eval_tfms)
ds_test            = datasets.ImageFolder(root=str(imagebits_test_dir),  transform=eval_tfms)

ds_train_base = Subset(ds_train_base_full, train_idx)
ds_train_aug  = Subset(ds_train_aug_full,  train_idx)
ds_val        = Subset(ds_val_full,        val_idx)


def make_loader(ds, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY and (device.type == "cuda"),
    )

train_loader_base = make_loader(ds_train_base, shuffle=True)
train_loader_aug  = make_loader(ds_train_aug,  shuffle=True)
val_loader        = make_loader(ds_val,        shuffle=False)
test_loader       = make_loader(ds_test,       shuffle=False)

xb, yb = next(iter(train_loader_base))
print("Baseline batch:", xb.shape, xb.dtype, "| y:", yb.shape, yb.dtype)


## Modele: MLP & CNN


In [ ]:
# =========================
# 7) Models
# =========================

class MLP(nn.Module):
    def __init__(self, in_shape=(3, 96, 96), num_classes=10, hidden_dims=(1024, 512), dropout=0.3):
        super().__init__()
        c, h, w = in_shape
        in_dim = c * h * w

        layers = []
        prev = in_dim
        for hd in hidden_dims:
            layers += [
                nn.Linear(prev, hd),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            prev = hd

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2),
            )

        self.features = nn.Sequential(
            block(3, 32),
            block(32, 64),
            block(64, 128),
            block(128, 256),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * (cfg.IMG_SIZE // 16) * (cfg.IMG_SIZE // 16), 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## Training utilities


In [ ]:
# =========================
# 8) Train / Eval loops
# =========================

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    all_preds = []
    all_true = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

        all_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        all_true.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds) if all_preds else np.array([])
    all_true = np.concatenate(all_true) if all_true else np.array([])

    return {
        "loss": total_loss / max(n_batches, 1),
        "acc": total_acc / max(n_batches, 1),
        "y_true": all_true,
        "y_pred": all_preds,
    }


def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and scaler.is_enabled():
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return {
        "loss": total_loss / max(n_batches, 1),
        "acc": total_acc / max(n_batches, 1),
    }


In [ ]:
# =========================
# 9) fit + plots
# =========================

def fit_model(
    model: nn.Module,
    train_loader,
    val_loader,
    epochs: int,
    lr: float,
    weight_decay: float,
    device,
    use_amp: bool = True,
):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler=scaler if use_amp else None)
        va = eval_one_epoch(model, val_loader, criterion, device)

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["acc"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["acc"])

        if va["acc"] > best_val_acc:
            best_val_acc = va["acc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {tr['loss']:.4f} acc {tr['acc']:.4f} | "
            f"val loss {va['loss']:.4f} acc {va['acc']:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val_acc


def plot_history(history, title_prefix=""):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    plt.figure()
    plt.plot(epochs, history["train_loss"], label="train")
    plt.plot(epochs, history["val_loss"], label="val")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(epochs, history["train_acc"], label="train")
    plt.plot(epochs, history["val_acc"], label="val")
    plt.title(f"{title_prefix} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_confusion(cm, class_names, title="Confusion matrix"):
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()


## Experimente

Rulăm 3 experimente:
1. **MLP Baseline** (fără aug)
2. **CNN Baseline** (fără aug)
3. **CNN Augmented** (train cu aug)



In [ ]:
# =========================
# 10) Experiments
# =========================
results = []

def eval_macro_f1(model, loader, device):
    crit = nn.CrossEntropyLoss()
    res = eval_one_epoch(model, loader, crit, device)
    y_true = res["y_true"]
    y_pred = res["y_pred"]
    return f1_score(y_true, y_pred, average="macro") if len(y_true) else 0.0

# --- MLP baseline ---
seed_everything(cfg.SEED)
mlp_base = MLP(in_shape=(3, cfg.IMG_SIZE, cfg.IMG_SIZE), num_classes=num_classes, hidden_dims=(1024, 512), dropout=0.3)
mlp_base, hist_mlp_base, best_val_mlp_base = fit_model(
    mlp_base,
    train_loader_base,
    val_loader,
    epochs=cfg.EPOCHS,
    lr=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
    device=device,
    use_amp=use_amp,
)
plot_history(hist_mlp_base, title_prefix="MLP Baseline (no aug)")
val_f1_mlp = eval_macro_f1(mlp_base, val_loader, device)
print("Best val acc (MLP baseline):", best_val_mlp_base, "| Val macro-F1:", val_f1_mlp)
results.append({
    "model": "MLP baseline",
    "arch_details": "Flatten -> Linear(1024) -> ReLU -> Dropout(0.3) -> Linear(512) -> ReLU -> Dropout(0.3) -> Linear(num_classes)",
    "aug": "no",
    "optimizer": "Adam",
    "lr": cfg.LR,
    "batch_size": cfg.BATCH_SIZE,
    "epochs": cfg.EPOCHS,
    "weight_decay": cfg.WEIGHT_DECAY,
    "best_val_acc": best_val_mlp_base,
    "val_macro_f1": val_f1_mlp,
})

# --- CNN baseline (no aug) ---
seed_everything(cfg.SEED)
cnn_base = SimpleCNN(num_classes=num_classes, dropout=0.4)
cnn_base, hist_cnn_base, best_val_cnn_base = fit_model(
    cnn_base,
    train_loader_base,
    val_loader,
    epochs=cfg.EPOCHS,
    lr=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
    device=device,
    use_amp=use_amp,
)
plot_history(hist_cnn_base, title_prefix="CNN Baseline (no aug)")
val_f1_cnn_base = eval_macro_f1(cnn_base, val_loader, device)
print("Best val acc (CNN baseline):", best_val_cnn_base, "| Val macro-F1:", val_f1_cnn_base)
results.append({
    "model": "CNN baseline",
    "arch_details": "Conv(3->32)->BN->ReLU->Pool x3 + Linear(256) + Dropout(0.4)",
    "aug": "no",
    "optimizer": "Adam",
    "lr": cfg.LR,
    "batch_size": cfg.BATCH_SIZE,
    "epochs": cfg.EPOCHS,
    "weight_decay": cfg.WEIGHT_DECAY,
    "best_val_acc": best_val_cnn_base,
    "val_macro_f1": val_f1_cnn_base,
})

# --- CNN augmented (train aug) ---
seed_everything(cfg.SEED)
cnn_aug = SimpleCNN(num_classes=num_classes, dropout=0.4)
cnn_aug, hist_cnn_aug, best_val_cnn_aug = fit_model(
    cnn_aug,
    train_loader_aug,
    val_loader,
    epochs=cfg.EPOCHS,
    lr=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
    device=device,
    use_amp=use_amp,
)
plot_history(hist_cnn_aug, title_prefix="CNN Augmented (train aug)")
val_f1_cnn_aug = eval_macro_f1(cnn_aug, val_loader, device)
print("Best val acc (CNN aug):", best_val_cnn_aug, "| Val macro-F1:", val_f1_cnn_aug)
results.append({
    "model": "CNN aug",
    "arch_details": "Same CNN as baseline; train-time augmentations",
    "aug": "yes",
    "optimizer": "Adam",
    "lr": cfg.LR,
    "batch_size": cfg.BATCH_SIZE,
    "epochs": cfg.EPOCHS,
    "weight_decay": cfg.WEIGHT_DECAY,
    "best_val_acc": best_val_cnn_aug,
    "val_macro_f1": val_f1_cnn_aug,
})

# --- comparație pe aceleași grafice ---
epochs = np.arange(1, len(hist_cnn_base["train_loss"]) + 1)

plt.figure()
plt.plot(epochs, hist_cnn_base["val_loss"], label="val loss (no aug)")
plt.plot(epochs, hist_cnn_aug["val_loss"],  label="val loss (aug)")
plt.title("CNN – Val Loss: no aug vs aug")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(epochs, hist_cnn_base["val_acc"], label="val acc (no aug)")
plt.plot(epochs, hist_cnn_aug["val_acc"],  label="val acc (aug)")
plt.title("CNN – Val Accuracy: no aug vs aug")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## Evaluare pe test set + Confusion Matrix

Selectăm cel mai bun CNN dintre baseline și augmented (după **best val acc**) și îl evaluăm pe **test**.


In [ ]:
# =========================
# 11) Test evaluation
# =========================

def evaluate_split(model, loader, device, class_names, title_prefix=""):
    criterion = nn.CrossEntropyLoss()
    res = eval_one_epoch(model, loader, criterion, device)

    y_true = res["y_true"]
    y_pred = res["y_pred"]

    acc = (y_true == y_pred).mean() if len(y_true) else 0.0
    f1  = f1_score(y_true, y_pred, average="macro") if len(y_true) else 0.0

    print(f"{title_prefix} accuracy: {acc:.4f}")
    print(f"{title_prefix} macro-F1 : {f1:.4f}")
    return acc, f1, y_true, y_pred

# Test scores pentru fiecare model
test_acc_mlp, test_f1_mlp, _, _ = evaluate_split(mlp_base, test_loader, device, class_names, title_prefix="MLP baseline (test)")
test_acc_cnn_base, test_f1_cnn_base, _, _ = evaluate_split(cnn_base, test_loader, device, class_names, title_prefix="CNN baseline (test)")
test_acc_cnn_aug,  test_f1_cnn_aug,  _, _ = evaluate_split(cnn_aug,  test_loader, device, class_names, title_prefix="CNN aug (test)")

# Selectăm best CNN după best val acc
best_cnn_model = cnn_aug if best_val_cnn_aug >= best_val_cnn_base else cnn_base
best_cnn_name  = "CNN Aug" if best_val_cnn_aug >= best_val_cnn_base else "CNN Baseline"

acc_best, f1_best, y_true, y_pred = evaluate_split(best_cnn_model, test_loader, device, class_names, title_prefix=f"{best_cnn_name} (selected test)")
print("\nClassification report (best CNN on test):")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm_cnn = confusion_matrix(y_true, y_pred)
plot_confusion(cm_cnn, class_names, title=f"{best_cnn_name} Confusion Matrix (test)")


## Tabel rezultate (pentru PDF)


In [ ]:
# =========================
# 12) Results table
# =========================

# Map test results
test_map = {
    "MLP baseline": (test_acc_mlp, test_f1_mlp),
    "CNN baseline": (test_acc_cnn_base, test_f1_cnn_base),
    "CNN aug":      (test_acc_cnn_aug, test_f1_cnn_aug),
}

for row in results:
    if row["model"] in test_map:
        row["test_acc"] = float(test_map[row["model"]][0])
        row["test_macro_f1"] = float(test_map[row["model"]][1])

df = pd.DataFrame(results)
display(df)

In [ ]:
import matplotlib.pyplot as plt

def plot_compare(hist_a, hist_b, label_a="No Aug", label_b="Aug", title="CNN Comparison"):
    epochs = range(1, len(hist_a["val_loss"]) + 1)

    # Val Loss comparison
    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a["val_loss"], label=f"{label_a} - Val Loss")
    plt.plot(epochs, hist_b["val_loss"], label=f"{label_b} - Val Loss")
    plt.title(title + " | Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    # Val Acc comparison
    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a["val_acc"], label=f"{label_a} - Val Acc")
    plt.plot(epochs, hist_b["val_acc"], label=f"{label_b} - Val Acc")
    plt.title(title + " | Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

plot_compare(hist_cnn_base, hist_cnn_aug, title="CNN: No Aug vs Aug")


In [ ]:
import torch

ckpt = {
    "model_name": "SimpleCNN",
    "img_size": CFG.IMG_SIZE,
    "num_classes": len(class_names),
    "class_names": class_names,
    "state_dict": cnn_aug.state_dict(),
}

torch.save(ckpt, "cnn_imagebits_64.pth")
print("Saved checkpoint: cnn_imagebits_64.pth")
